## Import

In [1]:
import json
import random
from pathlib import Path
from typing import Dict, List, Tuple

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, concatenate_datasets, load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

c:\Users\Anechka\Documents\Anya\лэти\itikipiz\lab6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Consts

In [2]:
SEED = 42
MAX_LENGTH = 128
BASE_MODEL_NAME = "Helsinki-NLP/opus-mt-en-ru"

OUTPUTS_DIR = Path("outputs")
MODEL_DIRS = {
    "model_news": OUTPUTS_DIR / "model_news",
    "model_books": OUTPUTS_DIR / "model_books",
    "model_combined": OUTPUTS_DIR / "model_combined",
}
METRICS_DIR = OUTPUTS_DIR / "metrics"
PREDICTIONS_DIR = OUTPUTS_DIR / "predictions"

DATASET_CONFIGS = {
    "news": {
        "path": "Helsinki-NLP/news_commentary",
        "lang_pair": "en-ru",
        "train_limit": 3000,
        "test_limit": 300,
    },
    "books": {
        "path": "Helsinki-NLP/opus_books",
        "lang_pair": "en-ru",
        "train_limit": 3000,
        "test_limit": 300,
    },
}

# Organization functions for enviroment, libs

In [3]:
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_output_dirs() -> None:
    OUTPUTS_DIR.mkdir(exist_ok=True)
    METRICS_DIR.mkdir(parents=True, exist_ok=True)
    PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
    for model_dir in MODEL_DIRS.values():
        model_dir.mkdir(parents=True, exist_ok=True)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
set_seed(SEED)
ensure_output_dirs()
print(f"Device: {get_device()}")

Device: cpu


# Work with data

In [5]:
def load_translation_dataset(dataset_path: str, lang_pair: str):
    last_error = None
    attempts = [
        lambda: load_dataset(dataset_path, lang_pair),
        lambda: load_dataset(dataset_path, name=lang_pair),
        lambda: load_dataset(dataset_path, lang_pair=lang_pair),
    ]
    for attempt in attempts:
        try:
            return attempt()
        except TypeError as error:
            last_error = error
        except Exception as error:
            last_error = error
    raise RuntimeError(f"Could not load dataset {dataset_path} ({lang_pair}): {last_error}")


def get_translation_column_name(dataset: Dataset) -> str:
    for candidate in ("translation", "translations"):
        if candidate in dataset.column_names:
            return candidate
    raise ValueError(f"Translation column not found in columns: {dataset.column_names}")


def normalize_translation_example(example: Dict, translation_column: str) -> Dict[str, str]:
    translation = example[translation_column]
    if isinstance(translation, list):
        translation = translation[0]
    if not isinstance(translation, dict):
        raise ValueError("Unexpected translation field format")
    return {"source_text": translation["en"], "target_text": translation["ru"]}


def build_small_splits(dataset_key: str) -> Tuple[Dataset, Dataset]:
    config = DATASET_CONFIGS[dataset_key]
    dataset_dict = load_translation_dataset(config["path"], config["lang_pair"])

    base_split_name = "train" if "train" in dataset_dict else list(dataset_dict.keys())[0]
    base_dataset = dataset_dict[base_split_name]
    translation_column = get_translation_column_name(base_dataset)

    normalized = base_dataset.map(
        lambda row: normalize_translation_example(row, translation_column),
        remove_columns=base_dataset.column_names,
    )
    normalized = normalized.shuffle(seed=SEED)

    sample_limit = min(len(normalized), config["train_limit"])
    sampled = normalized.select(range(sample_limit))
    split_dataset = sampled.train_test_split(test_size=0.2, seed=SEED)

    test_limit = min(len(split_dataset["test"]), config["test_limit"])
    train_dataset = split_dataset["train"]
    test_dataset = split_dataset["test"].select(range(test_limit))
    return train_dataset, test_dataset


# Tokenization

In [6]:
def tokenize_batch(batch: Dict[str, List[str]], tokenizer: AutoTokenizer) -> Dict[str, List[List[int]]]:
    model_inputs = tokenizer(
        batch["source_text"],
        max_length=MAX_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


def prepare_tokenized_datasets(
    train_dataset: Dataset,
    eval_dataset: Dataset,
    tokenizer: AutoTokenizer,
) -> Tuple[Dataset, Dataset]:
    tokenized_train = train_dataset.map(
        lambda batch: tokenize_batch(batch, tokenizer),
        batched=True,
        remove_columns=train_dataset.column_names,
    )
    tokenized_eval = eval_dataset.map(
        lambda batch: tokenize_batch(batch, tokenizer),
        batched=True,
        remove_columns=eval_dataset.column_names,
    )
    return tokenized_train, tokenized_eval


def load_model_and_tokenizer(model_path: str) -> Tuple[AutoModelForSeq2SeqLM, AutoTokenizer]:
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
    model.to(get_device())
    return model, tokenizer


# Translate

In [7]:
def translate_texts(
    texts: List[str],
    model: AutoModelForSeq2SeqLM,
    tokenizer: AutoTokenizer,
    batch_size: int = 16,
) -> List[str]:
    predictions = []
    model.eval()
    device = get_device()

    for start_idx in range(0, len(texts), batch_size):
        batch_texts = texts[start_idx : start_idx + batch_size]
        batch = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )
        batch = {key: value.to(device) for key, value in batch.items()}

        with torch.no_grad():
            generated_tokens = model.generate(
                **batch,
                max_length=MAX_LENGTH,
                num_beams=4,
            )
        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        predictions.extend(decoded)

    return predictions


# Training and savings

In [8]:
def save_predictions(
    model_name: str,
    test_dataset_name: str,
    sources: List[str],
    predictions: List[str],
    references: List[str],
    sample_count: int = 10,
) -> None:
    records = []
    for source, prediction, reference in zip(sources[:sample_count], predictions[:sample_count], references[:sample_count]):
        records.append(
            {
                "source_text": source,
                "prediction": prediction,
                "reference": reference,
            }
        )

    output_path = PREDICTIONS_DIR / f"{model_name}_{test_dataset_name}.json"
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(records, file, ensure_ascii=False, indent=2)





def train_model(output_dir: Path, train_dataset: Dataset, eval_dataset: Dataset) -> None:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL_NAME)

    tokenized_train, tokenized_eval = prepare_tokenized_datasets(train_dataset, eval_dataset, tokenizer)
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

    training_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=1,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=5e-5,
        weight_decay=0.01,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        predict_with_generate=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        processing_class=tokenizer,
        data_collator=data_collator,
    )
    trainer.train()

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)


# BLEU

In [9]:
def compute_bleu(predictions: List[str], references: List[str]) -> float:
    metric = evaluate.load("sacrebleu")
    result = metric.compute(
        predictions=predictions,
        references=[[reference] for reference in references],
    )
    return float(result["score"])

# Evaluate

In [10]:
def evaluate_model_on_dataset(
    model_name: str,
    model_path: str,
    test_dataset_name: str,
    test_dataset: Dataset,
) -> Dict[str, object]:
    model, tokenizer = load_model_and_tokenizer(model_path)
    sources = test_dataset["source_text"]
    references = test_dataset["target_text"]
    predictions = translate_texts(sources, model, tokenizer)
    bleu = compute_bleu(predictions, references)
    save_predictions(model_name, test_dataset_name, sources, predictions, references)
    return {
        "model_name": model_name,
        "test_dataset": test_dataset_name,
        "bleu": round(bleu, 4),
    }

# Pipeline

In [11]:
def run_baseline(sample_size: int = 100) -> pd.DataFrame:
    _, news_test = build_small_splits("news")
    sample_size = min(sample_size, len(news_test))
    news_sample = news_test.select(range(sample_size))
    result = evaluate_model_on_dataset("baseline", BASE_MODEL_NAME, "news_test", news_sample)
    results_df = pd.DataFrame([result])
    return results_df


def run_train_news() -> None:
    train_dataset, test_dataset = build_small_splits("news")
    train_model(MODEL_DIRS["model_news"], train_dataset, test_dataset)


def run_train_books() -> None:
    train_dataset, test_dataset = build_small_splits("books")
    train_model(MODEL_DIRS["model_books"], train_dataset, test_dataset)


def run_train_combined() -> None:
    news_train, news_test = build_small_splits("news")
    books_train, books_test = build_small_splits("books")
    combined_train = concatenate_datasets([news_train, books_train]).shuffle(seed=SEED)
    combined_eval = concatenate_datasets([news_test, books_test]).shuffle(seed=SEED)
    train_model(MODEL_DIRS["model_combined"], combined_train, combined_eval)


def resolve_local_model(path: Path) -> str:
    if path.exists():
        return str(path)
    raise FileNotFoundError(f"Model directory does not exist: {path}")


def run_evaluate_all() -> pd.DataFrame:
    _, news_test = build_small_splits("news")
    _, books_test = build_small_splits("books")

    local_combined_candidates = [
        MODEL_DIRS["model_combined"],
        OUTPUTS_DIR / "model_conbined",
    ]
    combined_path = None
    for candidate in local_combined_candidates:
        if candidate.exists():
            combined_path = str(candidate)
            break
    if combined_path is None:
        raise FileNotFoundError("Combined model not found. Run run_train_combined() first.")

    model_paths = {
        "baseline": BASE_MODEL_NAME,
        "news-finetuned": resolve_local_model(MODEL_DIRS["model_news"]),
        "books-finetuned": resolve_local_model(MODEL_DIRS["model_books"]),
        "model-finetuned": combined_path,
    }

    test_datasets = {
        "news_test": news_test,
        "books_test": books_test,
    }

    results = []
    for model_name, model_path in model_paths.items():
        for test_dataset_name, test_dataset in test_datasets.items():
            results.append(
                evaluate_model_on_dataset(
                    model_name=model_name,
                    model_path=model_path,
                    test_dataset_name=test_dataset_name,
                    test_dataset=test_dataset,
                )
            )

    results_df = pd.DataFrame(results, columns=["model_name", "test_dataset", "bleu"])
    results_df.to_csv(METRICS_DIR / "results.csv", index=False)
    return results_df


def run_translate(text: str, model_path: str) -> str:
    model, tokenizer = load_model_and_tokenizer(model_path)
    translated_text = translate_texts([text], model, tokenizer, batch_size=1)[0]
    return translated_text


## Baseline

Считает BLEU для исходной модели на небольшой выборке `news_commentary`.

In [12]:
baseline_results = run_baseline(sample_size=100)
baseline_results


c:\Users\Anechka\Documents\Anya\лэти\itikipiz\lab6\.venv\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 13126.12it/s]


,model_name,test_dataset,bleu
0,baseline,news_test,28.785


## Fine-tuning: news

In [13]:
run_train_news()


c:\Users\Anechka\Documents\Anya\лэти\itikipiz\lab6\.venv\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 11187.70it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
c:\Users\Anechka\Documents\Anya\лэти\itikipiz\lab6\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.479784,1.450136


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.05it/s]


## Fine-tuning: books

In [14]:
run_train_books()


c:\Users\Anechka\Documents\Anya\лэти\itikipiz\lab6\.venv\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 14433.79it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
c:\Users\Anechka\Documents\Anya\лэти\itikipiz\lab6\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.848755,1.859719


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.05it/s]


## Fine-tuning: combined

In [15]:
run_train_combined()


c:\Users\Anechka\Documents\Anya\лэти\itikipiz\lab6\.venv\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 28228.26it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
c:\Users\Anechka\Documents\Anya\лэти\itikipiz\lab6\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.753279,1.660876


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.89it/s]


## Evaluate All Models

После обучения всех моделей можно получить таблицу BLEU и сохранить ее в `outputs/metrics/results.csv`.

In [16]:
all_results = run_evaluate_all()
all_results


c:\Users\Anechka\Documents\Anya\лэти\itikipiz\lab6\.venv\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 15010.83it/s]
Using the latest cached version of the module from C:\Users\Anechka\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--sacrebleu\28676bf65b4f88b276df566e48e603732d0b4afd237603ebdf92acaacf5be99b (last modified on Fri Apr 24 23:15:38 2026) since it couldn't be found locally at evaluate-metric--sacrebleu, or remotely on the Hugging Face Hub.
Loading weights: 100%|██████████| 254/254 [00:00<00:00, 5018.43it/s]


,model_name,test_dataset,bleu
0,baseline,news_test,27.9259
1,baseline,books_test,18.9065
2,news-finetuned,news_test,28.2256
3,news-finetuned,books_test,17.9154
4,books-finetuned,news_test,25.5738
5,books-finetuned,books_test,22.3986
6,model-finetuned,news_test,27.9739
7,model-finetuned,books_test,21.9272


## Translate One Sentence

In [17]:
run_translate("Machine translation is useful.", "outputs/model_news")


c:\Users\Anechka\Documents\Anya\лэти\itikipiz\lab6\.venv\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 254/254 [00:00<00:00, 5352.70it/s]


'Машинный перевод полезен.'